# GoalAnalytics — World Cup 2026 AI Prediction Engine
**Author:** GoalAnalytics  
**Date:** June 2026  
**Repository:** [github.com/nithinnarla/goal-analytics](https://github.com/nithinnarla/goal-analytics)

---
## Overview

This notebook runs a complete end-to-end analysis of the 2026 FIFA World Cup using:
- **Historical data** (1930–2022) from martj42/international_results
- **Elo ratings** computed from 150+ years of international football results
- **Bivariate Poisson model** (Dixon-Coles methodology) for scoreline distributions
- **Logistic Regression** (scikit-learn) trained on WC historical outcomes
- **Monte Carlo simulation** (10,000 runs) for tournament win probabilities

Matches `GoalAnalytics_Analysis.py` cell-for-cell.


## 0. Setup & Imports

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))   # so we can import from goal-analytics/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

print("Dependencies loaded ✓")
print(f"  numpy  {np.__version__}")
print(f"  pandas {pd.__version__}")
print(f"  seaborn {sns.__version__}")


## 1. Historical World Cup Data (1930–2022)

Source: [martj42/international_results](https://github.com/martj42/international_results) — the most comprehensive public football results dataset.  
No authentication required.

> ⚠️  World Cup started in **1930** (Uruguay). There was no 1932 World Cup.
> Tournaments held every 4 years: 1930, 1934, 1938, 1950, 1954, …, 2022.


In [ ]:
from data.historical import (
    fetch_results, get_wc_matches, get_wc_stats,
    compute_elo_ratings, build_form_lookup,
    WC_WINNERS, WC_HOSTS,
)

results_df = fetch_results()
wc_df      = get_wc_matches(results_df)
wc_stats   = get_wc_stats(wc_df)

if wc_df is not None:
    print(f"WC matches in dataset: {len(wc_df)}")
    print(f"Years covered: {sorted(set(wc_df['date'].dt.year))}")
else:
    print("Offline mode: using built-in historical stats")

print(f"\nWC title counts:")
for team, titles in sorted(wc_stats['titles'].items(), key=lambda x: -x[1]):
    print(f"  {team:<15} {titles}")


### Figure 01 — Historical WC Title Counts

In [ ]:
from analysis.visualizations import plot_wc_winners_history, save_fig

fig01 = plot_wc_winners_history(wc_stats)
save_fig(fig01, "01_wc_winners_history.png")
fig01


### Figure 02 — Avg Goals per Game by Year

In [ ]:
from analysis.visualizations import plot_goals_per_tournament

fig02 = plot_goals_per_tournament(wc_stats)
save_fig(fig02, "02_goals_per_tournament.png")
fig02


## 2. Elo Ratings — Computed from 150 Years of Results

The Elo system rates teams by replaying every match chronologically:
- **K = 40** for World Cup final tournament matches  
- **K = 25** for other competitive fixtures (qualifiers, continental cups)  
- **K = 10** for friendlies  
- **Goal margin bonus:** 1-goal = 1×K · 2-goal = 1.5×K · 3+ = 1.75×K  
- **Home advantage:** +100 Elo applied to the home team before computing expected score  

We **blend** historical Elo (60%) with hand-calibrated preset Elo (40%) for the 2026 model.


In [ ]:
from data.teams import TEAMS, get_elo as preset_get_elo

if results_df is not None:
    historical_elos = compute_elo_ratings(results_df)
    blended_elos = {
        t: 0.6 * preset_get_elo(t) + 0.4 * historical_elos.get(t, preset_get_elo(t))
        for t in TEAMS
    }
    print(f"Elo computed from {len(results_df):,} matches")
else:
    blended_elos = {t: preset_get_elo(t) for t in TEAMS}
    print("Using preset Elo ratings")

top15 = sorted(blended_elos.items(), key=lambda x: -x[1])[:15]
print("\nTop 15 Elo Ratings:")
for i, (team, elo) in enumerate(top15, 1):
    flag = TEAMS.get(team, {}).get("flag", "")
    print(f"  {i:2}. {flag} {team:<25} {elo:.0f}")


## 3. Feature Engineering & Logistic Regression

### Features
| Feature | Description |
|---------|-------------|
| `elo_diff` | home_elo − away_elo |
| `home_advantage` | 1 if home match, 0 if neutral |
| `form_diff` | home avg pts/game − away avg pts/game (last 15 matches) |
| `scored_diff` | home avg goals − away avg goals |
| `conceded_diff` | away avg conceded − home avg conceded |
| `elo_sq_diff` | elo_diff² / 10000 (non-linearity) |

### Model
Multinomial Logistic Regression (`sklearn.linear_model.LogisticRegression`)  
Trained on all WC final-tournament matches 1930–2022 (~900 matches)  
5-fold stratified cross-validation


In [ ]:
from models.logistic import LogisticMatchPredictor
from models.features import build_match_features_df

logistic_model = LogisticMatchPredictor()

if wc_df is not None and results_df is not None:
    form_lookup = build_form_lookup(results_df, as_of="2026-06-01", n=15)
    X, y = build_match_features_df(wc_df, blended_elos, form_lookup)
    logistic_model.train(X, y, cv=5)
    print(logistic_model.summary())

    # Feature importances
    print("\nFeature coefficients (Home Win class):")
    fi = logistic_model.feature_importance()
    if fi:
        from models.features import FEATURE_COLS
        hw_coefs = fi.get("home_win", [])
        for feat, coef in zip(FEATURE_COLS, hw_coefs):
            print(f"  {feat:<20} {coef:+.4f}")
else:
    form_lookup = {}
    print("Offline: logistic model using Elo fallback")


## 4. All 72 Group Stage Match Predictions

Both models are applied to every group stage fixture.  
`predicted_score` = most likely scoreline from Bivariate Poisson distribution.


In [ ]:
from data.fixtures import FIXTURES
from data import GROUPS
from models.elo import match_probabilities
from models.poisson import most_likely_score, expected_goals

predictions = {}
for fixture in FIXTURES:
    elo_h = blended_elos.get(fixture.home, preset_get_elo(fixture.home))
    elo_a = blended_elos.get(fixture.away, preset_get_elo(fixture.away))

    elo_probs  = match_probabilities(elo_h, elo_a)
    xg_h, xg_a = expected_goals(elo_h, elo_a)
    sh, sa     = most_likely_score(xg_h, xg_a)

    hf = form_lookup.get(fixture.home, {})
    af = form_lookup.get(fixture.away, {})
    log_probs = logistic_model.predict(
        home_elo=elo_h, away_elo=elo_a,
        home_form=hf.get("form", 1.5), away_form=af.get("form", 1.5),
        home_scored=hf.get("avg_scored", 1.3), away_scored=af.get("avg_scored", 1.3),
        home_conceded=hf.get("avg_conceded", 1.1), away_conceded=af.get("avg_conceded", 1.1),
    )

    predictions[fixture.match_id] = {
        "group": fixture.group, "home": fixture.home, "away": fixture.away,
        "xg_home": round(xg_h, 2), "xg_away": round(xg_a, 2),
        "predicted_score": f"{sh}-{sa}",
        "elo_home_win": round(elo_probs["home_win"], 3),
        "elo_draw":     round(elo_probs["draw"], 3),
        "elo_away_win": round(elo_probs["away_win"], 3),
        "log_home_win": round(log_probs["home_win"], 3),
        "log_draw":     round(log_probs["draw"], 3),
        "log_away_win": round(log_probs["away_win"], 3),
    }

preds_df = pd.DataFrame.from_dict(predictions, orient="index")
print(f"Total matches predicted: {len(preds_df)}")
preds_df.head(6)


### All Matches by Group

In [ ]:
for grp in sorted(GROUPS.keys()):
    group_preds = preds_df[preds_df["group"] == grp]
    print(f"\n─── Group {grp} ────────────────────────────────────────")
    print(f"  {'Home':<22} vs {'Away':<22} Score  ELO-HW   ELO-D  ELO-AW  LOG-HW   LOG-AW")
    for _, row in group_preds.iterrows():
        print(f"  {row['home']:<22} vs {row['away']:<22} {row['predicted_score']:<6} "
              f"{row['elo_home_win']:.2f}  {row['elo_draw']:.2f}  {row['elo_away_win']:.2f}  "
              f"{row['log_home_win']:.2f}   {row['log_away_win']:.2f}")


## 5. Monte Carlo Simulation — Tournament Win Probabilities

**10,000 independent full-tournament simulations.**

Each simulation:
1. Group stage: 48 teams → sample results from Poisson distributions
2. Apply FIFA tiebreaker rules (pts → GD → GF → H2H → random)
3. Pick top 2 from each group + 8 best 3rd-place teams → R32
4. Simulate knockout rounds (draw brackets per 2026 WC format)
5. Track final winner


In [ ]:
from models.monte_carlo import run_simulations, simulate_group

sim_n = 10_000
n_group_sims = 2_000
print(f"Running {sim_n:,} tournament simulations...")
round_probs = run_simulations(n=sim_n, known_results={}, seed=42)
win_probs = dict(sorted(
    ((team, p.get("Winner", 0.0)) for team, p in round_probs.items()),
    key=lambda x: x[1], reverse=True
))

# Group finish distributions — simulate each group independently
print(f"Computing group finish distributions ({n_group_sims} sims/group)...")
group_finish = {}
for grp in sorted(GROUPS.keys()):
    for _ in range(n_group_sims):
        ranked = simulate_group(grp, {})
        for pos, rec in enumerate(ranked, 1):
            group_finish.setdefault(rec.team, {})[pos] = group_finish.get(rec.team, {}).get(pos, 0) + 1

for team in group_finish:
    total = sum(group_finish[team].values())
    group_finish[team] = {k: v/total for k, v in group_finish[team].items()}

print("\nTop 15 Tournament Win Probabilities:")
for i, (team, p) in enumerate(list(win_probs.items())[:15], 1):
    flag = TEAMS.get(team, {}).get("flag", "")
    bar = "█" * int(p * 200)
    print(f"  {i:2}. {flag} {team:<22} {p*100:5.1f}%  {bar}")


### Figure 03 — All 12 Group Stage Predicted Standings

In [ ]:
from analysis.visualizations import plot_group_standings

fig03 = plot_group_standings(group_finish)
save_fig(fig03, "03_all_group_standings.png")
fig03


#### Individual Group Charts (A–L)

In [ ]:
from analysis.visualizations import plot_group_standings

for grp in sorted(GROUPS.keys()):
    fig = plot_group_standings(group_finish, group=grp)
    save_fig(fig, f"03_{grp}_group_standings.png")
    plt.show(fig)
    plt.close(fig)


### Figure 04 — Tournament Win Probabilities (Top 20)

In [ ]:
from analysis.visualizations import plot_win_probabilities

fig04 = plot_win_probabilities(win_probs, top_n=20)
save_fig(fig04, "04_win_probabilities.png")
fig04


### Figure 04b — Round-by-Round Progression (Top 8 Teams)

Figure 04 shows each team's *overall* title probability as a single bar — it
can't show **where** a team is most likely to fall short. This chart plots
P(reach R32 / R16 / QF / SF / Final / Win) for the top 8 contenders, using the
same `run_simulations()` output as Figure 04.

This is the figure that directly answers "is the Argentina–France final a
fluke?" — it shows Argentina, Spain, and France all clear ~75-80% to reach
the semifinals before separating in the final two rounds.

In [ ]:
from analysis.visualizations import plot_round_progression

fig04b = plot_round_progression(round_probs, top_n=8)
save_fig(fig04b, "04b_round_progression.png")
fig04b


## 6. Match Predictions — Showcase: Argentina vs France

The most anticipated potential final. Elo difference is small (~50 pts),
so expect high uncertainty with a slight Argentine edge at a neutral venue.


In [ ]:
elo_arg = blended_elos.get("Argentina", 2100)
elo_fra = blended_elos.get("France",    2050)

elo_p = match_probabilities(elo_arg, elo_fra)

arg_form = form_lookup.get("Argentina", {}).get("form", 1.7) if form_lookup else 1.7
fra_form = form_lookup.get("France",    {}).get("form", 1.6) if form_lookup else 1.6

log_p = logistic_model.predict(
    home_elo=elo_arg, away_elo=elo_fra,
    home_form=arg_form, away_form=fra_form,
    is_neutral=True,
)

# --- Random Forest & XGBoost, trained on recent-era results (2010-present) ---
from models.random_forest import RandomForestMatchPredictor
from models.xgboost_model import XGBoostMatchPredictor
from models.backtest import RECENT_ERA_START, _monte_carlo_match_probs

if results_df is not None:
    recent_df = results_df[results_df["date"] >= RECENT_ERA_START]
else:
    recent_df = None

if recent_df is not None and len(recent_df) >= 200 and blended_elos:
    rf_model  = RandomForestMatchPredictor.train_from_history(recent_df, blended_elos, form_lookup)
    xgb_model = XGBoostMatchPredictor.train_from_history(recent_df, blended_elos, form_lookup)
else:
    rf_model, xgb_model = RandomForestMatchPredictor(), XGBoostMatchPredictor()

rf_probs = rf_model.predict(
    home_elo=elo_arg, away_elo=elo_fra, is_neutral=True,
    home_form=arg_form, away_form=fra_form,
)
xgb_probs = xgb_model.predict(
    home_elo=elo_arg, away_elo=elo_fra, is_neutral=True,
    home_form=arg_form, away_form=fra_form,
)

# Monte Carlo: empirical win/draw/loss frequencies from scoreline sampling
mc_probs = _monte_carlo_match_probs(elo_arg, elo_fra, n_sims=5000)

print("Argentina vs France — Predicted Neutral Venue")
print(f"{'Outcome':<14}{'Elo+Poisson':>13}{'Logistic':>11}{'RF':>9}{'XGBoost':>10}{'MonteCarlo':>12}")
print("-" * 69)
for outcome in ["home_win", "draw", "away_win"]:
    label = {"home_win": "Argentina Win", "draw": "Draw", "away_win": "France Win"}[outcome]
    print(f"{label:<14}{elo_p[outcome]*100:>12.1f}%{log_p[outcome]*100:>10.1f}%"
          f"{rf_probs[outcome]*100:>8.1f}%{xgb_probs[outcome]*100:>9.1f}%{mc_probs[outcome]*100:>11.1f}%")

print(f"\nRandom Forest trained: {rf_model.trained}"
      + (f"  (CV accuracy: {rf_model.cv_accuracy:.1%})" if rf_model.trained and rf_model.cv_accuracy else " — using Elo fallback"))
print(f"XGBoost trained:       {xgb_model.trained}"
      + (f"  (CV accuracy: {xgb_model.cv_accuracy:.1%})" if xgb_model.trained and xgb_model.cv_accuracy else " — using Elo fallback"))

xg_h, xg_a = expected_goals(elo_arg, elo_fra)
print(f"\nExpected Goals: Argentina {xg_h:.2f} — France {xg_a:.2f}")
sh, sa = most_likely_score(xg_h, xg_a)
print(f"Most Likely Score: {sh}-{sa}")


### Figure 05 — Scoreline Heatmap: Argentina vs France

In [ ]:
from analysis.visualizations import plot_scoreline_heatmap

fig05 = plot_scoreline_heatmap("Argentina", "France", elo_arg, elo_fra)
save_fig(fig05, "05_argentina_vs_france_scoreline.png")
fig05


### Figure 06 — Model Comparison: Elo+Poisson, Logistic, Random Forest, XGBoost & Monte Carlo

Five independent models score the same Argentina vs France matchup. Random
Forest and XGBoost are trained on 2010–present results (when training data is
available); the Monte Carlo bars are empirical win/draw/loss frequencies from
sampling the Poisson scoreline distribution 5,000 times. Broad agreement
across models increases confidence; large divergences flag matches where the
models "see" the game differently.


In [ ]:
from analysis.visualizations import plot_model_comparison

fig06 = plot_model_comparison(
    "Argentina", "France", elo_p, log_p,
    rf_probs=rf_probs, xgb_probs=xgb_probs, mc_probs=mc_probs,
)
save_fig(fig06, "06_model_comparison_arg_fra.png")
fig06


## 6b. Model Backtest — Validating Against 2018 & 2022 World Cups

Before trusting any of the models above on the 2026 tournament, check how they
would have performed on the **last two** World Cups. Each model is retrained
using only data available *before* that tournament's kickoff (no lookahead),
then scored against the actual group-stage results:

- **Accuracy** — % of matches where the model's most likely outcome (home win
  / draw / away win) matched the real result.
- **Brier score** — mean squared error of the predicted probabilities vs. the
  actual outcome (0 = perfect; lower is better — this rewards well-calibrated
  probabilities, not just correct picks).

This requires fetching historical match data over the network. If the sandbox
running this notebook is offline, the cells below report that explicitly
rather than failing silently.


In [ ]:
from models.backtest import run_full_backtest
from analysis.visualizations import plot_backtest_results

print("Running point-in-time backtest for the 2018 and 2022 World Cups...")
print("(retrains Elo / Logistic / RF / XGBoost as of each tournament's kickoff date)\n")

backtest_results = run_full_backtest(years=(2018, 2022), n_mc_sims=5000)

for year in sorted(backtest_results):
    res = backtest_results[year]
    if "error" in res:
        print(f"WC {year}: \u26a0 {res['error']}")
        continue
    print(f"\nWC {year} — {res['n_matches']} group-stage matches "
          f"(trained on {res['n_train_wc']} prior WC + {res['n_train_recent']} recent matches)")
    print(f"{'Model':<26}{'Accuracy':>10}{'Brier':>10}{'n':>6}")
    for name, stats in res["models"].items():
        print(f"{name:<26}{stats['accuracy']*100:>9.1f}%{stats['brier']:>10.4f}{stats['n']:>6}")


### Figure 06b — Backtest Accuracy by Model


In [ ]:
fig06b = plot_backtest_results(backtest_results)
save_fig(fig06b, "06b_model_backtest.png")
fig06b


## 7. Knockout Stage Bracket — R32 → Final

This bracket comes from a single full-tournament simulation using
`models.monte_carlo.simulate_full_tournament_detailed()` — the same engine
behind the dashboard's Bracket tab. It plays out all 12 groups, applies FIFA's
official Annex C rules for slotting the 8 best third-placed teams into the
Round of 32, then simulates every knockout match (including extra time and
penalty shootouts where applicable) through to the Final and the 3rd-place
play-off.

> A single simulation is one possible path through the tournament, not the
> model's "average" prediction. The champion below can differ from the
> **aggregate** win probabilities in Figure 04 (Section 5), which are averaged
> over 10,000 simulations. The next cell reseeds `random` each time it runs,
> so re-running it will reproduce the same bracket — change the seed to see a
> different one.


In [ ]:
import random
from models.monte_carlo import simulate_full_tournament_detailed
from analysis.visualizations import build_bracket_from_simulation

random.seed(42)
rounds_reached, match_results = simulate_full_tournament_detailed()
bracket = build_bracket_from_simulation(match_results)

rounds = [("r32","Round of 32"), ("r16","Round of 16"), ("qf","Quarterfinals"),
           ("sf","Semifinals"), ("final","Final"), ("winner","\U0001f3c6 CHAMPION")]

for key, label in rounds:
    teams = bracket.get(key, [])
    print(f"\n  {label} ({len(teams)} teams):")
    for t in teams:
        p = win_probs.get(t, 0)
        flag = TEAMS.get(t, {}).get("flag", "")
        print(f"    {flag} {t:<22} {p*100:5.1f}%  (Fig.04 aggregate win prob)")

if "third_place" in bracket:
    bronze, fourth = bracket["third_place"]
    print(f"\n  \U0001f949 3rd Place: {TEAMS.get(bronze, {}).get('flag', '')} {bronze}")
    print(f"  4th Place:    {TEAMS.get(fourth, {}).get('flag', '')} {fourth}")


### Figure 07 — Predicted Knockout Bracket

In [ ]:
from analysis.visualizations import plot_knockout_bracket

fig07 = plot_knockout_bracket(bracket)
save_fig(fig07, "07_knockout_bracket.png")
fig07


## 8. Host Nation Advantage Analysis

Mexico, USA, and Canada each receive **+100 Elo** when playing in their
home cities. This is derived from empirical home-field advantage estimates
in tournament football (~0.3–0.4 goals per game advantage, translating to
roughly +100 Elo for international matches).


In [ ]:
from analysis.visualizations import plot_host_advantage

fig08 = plot_host_advantage()
save_fig(fig08, "08_host_advantage.png")
fig08


## Summary

| Metric | Value |
|--------|-------|
| Historical matches fetched | ~50,000+ (martj42 dataset) |
| WC matches trained on | ~900 (1930–2022) |
| Tournament simulations (Fig. 04 win probabilities) | 10,000 |
| Models compared (Fig. 06) | Elo+Poisson, Logistic Regression, Random Forest, XGBoost, Monte Carlo |
| Backtest coverage (Fig. 06b) | 2018 & 2022 World Cups, point-in-time retraining |
| Features (logistic / RF / XGBoost models) | 6 |
| Figures generated | 20+ |

### Predicted Champion

Two different things are reported in this notebook and shouldn't be confused:

- **Aggregate favorite (Figure 04)** — across 10,000 Monte Carlo simulations,
  Argentina has the highest tournament win probability (~28%), with Spain
  (~19%) and France (~16%) the main challengers.
- **Single-simulation bracket (Figure 07, Section 7)** — one specific
  full-tournament playthrough, including knockout-stage variance (extra time,
  penalties). Its champion is whichever team won *that particular*
  simulation, printed above — it may or may not be Argentina.

### Methodology Stack
```
Historical Data (martj42/GitHub)
        ↓
Elo Ratings (computed + preset blend)
        ↓
     ┌──────────────────────────────┐
     │  Bivariate Poisson           │  ← scoreline distributions, xG
     │  (Dixon-Coles 1997)          │
     └──────────────────────────────┘
     ┌──────────────────────────────┐
     │  Logistic Regression /       │  ← calibrated outcome probabilities
     │  Random Forest / XGBoost     │     (Fig. 06 model comparison)
     │  (sklearn, 5-fold CV)        │
     └──────────────────────────────┘
        ↓
Monte Carlo Simulation (10,000 runs → Fig. 04;
1 detailed run w/ real Annex-C bracket → Fig. 07)
        ↓
Tournament Win Probabilities + Predicted Bracket
        ↓
Backtest vs. 2018 & 2022 World Cups (Fig. 06b)
```


In [ ]:
champion = bracket["winner"][0] if bracket["winner"] else "N/A"
flag  = TEAMS.get(champion, {}).get("flag", "\U0001f3c6")
p_agg = win_probs.get(champion, 0)

agg_leader = max(win_probs, key=win_probs.get)
agg_flag   = TEAMS.get(agg_leader, {}).get("flag", "\U0001f3c6")
agg_p      = win_probs[agg_leader]

print(f"{'='*60}")
print(f"  {flag} Single-Simulation Bracket Champion: {champion}")
print(f"      (this team's aggregate win probability: {p_agg*100:.1f}%)")
print(f"\n  {agg_flag} Aggregate Win-Probability Leader (Fig. 04): {agg_leader} ({agg_p*100:.1f}%)")
if logistic_model.trained and logistic_model.cv_accuracy:
    print(f"\n  LogReg CV Accuracy:        {logistic_model.cv_accuracy:.1%}")
if rf_model.trained and rf_model.cv_accuracy:
    print(f"  Random Forest CV Accuracy: {rf_model.cv_accuracy:.1%}")
if xgb_model.trained and xgb_model.cv_accuracy:
    print(f"  XGBoost CV Accuracy:       {xgb_model.cv_accuracy:.1%}")
print(f"\n  All figures saved to: ../figures/")
print(f"{'='*60}")
